# Llama-3.2-1B — autoLRP attribution

Meta's Llama-3.2-1B (1.2B parameters): RMSNorm + RoPE + Grouped-Query Attention + SwiGLU MLP. The fused SDPA kernel needs no special setting: autoLRP decomposes it during the forward, or handles the fused node with the same rules.

**HuggingFace token required** — Llama is a gated repo.

In [ ]:
import os
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import autolrp
from autolrp import LRPConfig, explain, explain_summary
from _common import show_text_attribution, SHOWCASE_PROMPTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'meta-llama/Llama-3.2-1B'
MODEL_NAME = 'Llama-3.2-1B'
FILE_STEM = 'llama'

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float32, 
).eval().to(device)

class LlamaFromEmb(torch.nn.Module):
    """Embeddings → next-token logits (mean-centered for LRP)."""
    def __init__(self, m):
        super().__init__()
        self.body = m.model  # LlamaModel handles RoPE/RMSNorm/GQA internally
        self.head = m.lm_head
    def forward(self, inputs_embeds):
        h = self.body(inputs_embeds=inputs_embeds).last_hidden_state
        logits = self.head(h)[:, -1, :]
        return logits

wrapper = LlamaFromEmb(model).eval().to(device)

In [ ]:
for pi, text in enumerate(SHOWCASE_PROMPTS):
    ids = tok(text, return_tensors='pt').input_ids.to(device)
    emb = model.model.embed_tokens(ids).detach()

    with torch.no_grad():
        pred = wrapper(emb).argmax(-1).item()
    predicted = tok.decode([pred])

    x = autolrp.tensor(emb)
    out = wrapper(x)
    out[0, pred].lrp()                      # BASE: epsilon on linear layers, the RMSNorm statistic detached

    tokens = tok.convert_ids_to_tokens(ids[0])
    rel = x.relevance[0].sum(-1).detach().cpu().numpy()
    show_text_attribution(tokens, rel, predicted_token=predicted,
                         prompt_text=f'{MODEL_NAME}: "{text}" \u2192 "{predicted}"')

## What ran

RoPE multiplies by constant tables (relevance passes through), the RMSNorm scale is a statistic of its input (detached by `BASE`), the SwiGLU gate is a product of two live operands (proportional split).

In [ ]:
rows = explain(wrapper(autolrp.tensor(emb))[0, pred], LRPConfig())
print(explain_summary([r for r in rows if r[2] != 'native gradient']))